# Langfuse v3 — Dataset & Run Explorer / Exporter (REST + Judge edition)

Explore datasets, runs, run items, **LLM-as-a-judge scores + reasoning + the judge's own input prompt**,
then export to **Excel** and **JSON** — with full control over which fields appear.

Talks to the Langfuse **Public REST API** directly via `httpx` (no SDK, no OTel) so the
self-signed / TLS-proxy SSL issues don't apply.

### Where judge data lives (important)
- The judge's **score value** and **reasoning** are on the trace's `scores[]` → `value` + `comment`.
- The judge's **actual input prompt** is NOT on the score. Each judge run creates its own
  *execution trace* (environment `langfuse-llm-as-a-judge`). The score links to it via an
  execution-trace id, and that trace's `input` is the exact prompt the judge received.
- This notebook walks: run item → main trace → scores (value+reasoning) → judge execution
  trace (prompt + raw judge response), so you can confirm what each score was based on.

## 1. Connection (REST client)

In [ ]:
import httpx

LANGFUSE_HOST       = "https://your-langfuse.internal"   # no trailing slash
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxx"
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxx"

_client = httpx.Client(
    base_url=LANGFUSE_HOST,
    auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
    verify=False,
    trust_env=False,
    timeout=httpx.Timeout(30.0, read=120.0),
)

def lf_get(path, **params):
    r = _client.get(path, params={k: v for k, v in params.items() if v is not None})
    r.raise_for_status()
    return r.json()

h = _client.get("/api/public/health"); h.raise_for_status()
print("Connected OK ->", LANGFUSE_HOST, "|", h.json())

## 2. Field selection — choose what ends up in the output

This is your control panel. Set any field to `False` to drop it from BOTH the Excel sheet
and the flat JSON. The nested JSON always keeps everything (it's your raw archive).

`JUDGE_*` toggles control how much judge detail you pull. `PER_EVALUATOR_COLUMNS=True`
pivots each evaluator into its own columns (`score__<name>`, `reasoning__<name>`); set it
`False` to instead get one long `scores_json` blob column.

In [ ]:
# ---- run-item / trace fields ----
FIELDS = {
    "run_name":          True,
    "run_item_id":       True,
    "dataset_item_id":   True,
    "trace_id":          True,
    "observation_id":    False,
    "input":             True,   # application input (what your app received)
    "actual_output":     True,   # application output (what your app produced)
    "expected_output":   True,   # from the dataset item
    "reasoning_chain":   False,  # concatenated app-side observation outputs
    "n_observations":    False,
    "trace_name":        False,
    "latency_ms":        False,
    "total_cost":        False,
}

# ---- judge / evaluator behaviour ----
JUDGE_INCLUDE          = True   # fetch scores at all
JUDGE_FETCH_REASONING  = True   # include score.comment (the judge's reasoning)
JUDGE_FETCH_PROMPT     = True   # follow each score to its execution trace -> judge's input prompt + raw response
PER_EVALUATOR_COLUMNS  = True   # True: score__X / reasoning__X / judge_prompt__X columns
                                # False: single scores_json column with everything

INCLUDE_APP_OBSERVATIONS = False  # pull application-side observations for reasoning_chain (slower)
TEXT_LIMIT = 8000                 # truncate long cells for Excel (None = no truncation)

print("Field selection set. Active trace fields:",
      [k for k, v in FIELDS.items() if v])
print("Judge:", {"scores": JUDGE_INCLUDE, "reasoning": JUDGE_FETCH_REASONING,
                 "prompt": JUDGE_FETCH_PROMPT, "per_evaluator_cols": PER_EVALUATOR_COLUMNS})

## 3. Imports & helpers

In [ ]:
import json, datetime as dt
from pathlib import Path
from urllib.parse import quote
import pandas as pd

OUT_DIR = Path("langfuse_exports"); OUT_DIR.mkdir(exist_ok=True)
def _ts(): return dt.datetime.now().strftime("%Y%m%d_%H%M%S")

def _flatten(value, max_len=None):
    if value is None: return None
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False, default=str)
    value = str(value)
    if max_len and len(value) > max_len:
        value = value[:max_len] + " ...[truncated]"
    return value

def paginate(path, data_key="data", **params):
    page, out = 1, []
    while True:
        js = lf_get(path, page=page, limit=100, **params)
        data = js.get(data_key, []) or []
        if not data: break
        out.extend(data)
        meta = js.get("meta") or {}
        tp = meta.get("totalPages")
        if tp is not None and page >= tp: break
        if tp is None and len(data) < 100: break
        page += 1
    return out

def _apply_fields(row):
    """Keep only FIELDS that are True, preserving order."""
    return {k: row.get(k) for k, v in FIELDS.items() if v}

print("Helpers ready. Exports ->", OUT_DIR.resolve())

## 4. List all datasets

In [ ]:
datasets = paginate("/api/public/v2/datasets")
df_datasets = pd.DataFrame([{
    "name": d.get("name"), "id": d.get("id"),
    "description": d.get("description"),
    "metadata": _flatten(d.get("metadata")),
    "createdAt": d.get("createdAt"), "updatedAt": d.get("updatedAt"),
} for d in datasets])
print(f"{len(df_datasets)} dataset(s)\n")
df_datasets

## 5. Pick a dataset

In [ ]:
DATASET_NAME = "your-dataset-name"   # <-- edit me
dataset = lf_get(f"/api/public/v2/datasets/{quote(DATASET_NAME, safe='')}")
print("Dataset:", dataset.get("name"), "| id:", dataset.get("id"))

## 6. Mode A — Dataset items only
Filtered by `datasetName` (the `datasetId` filter is silently ignored by the public API).

In [ ]:
items = paginate("/api/public/dataset-items", datasetName=DATASET_NAME)
df_items = pd.DataFrame([{
    "item_id": it.get("id"), "status": it.get("status"),
    "input": _flatten(it.get("input")),
    "expected_output": _flatten(it.get("expectedOutput")),
    "metadata": _flatten(it.get("metadata")),
    "source_trace_id": it.get("sourceTraceId"),
    "createdAt": it.get("createdAt"),
} for it in items])
print(f"{len(df_items)} item(s)")
df_items.head(20)

## 7. List runs for this dataset

In [ ]:
runs = paginate(f"/api/public/datasets/{quote(DATASET_NAME, safe='')}/runs")
df_runs = pd.DataFrame([{
    "run_name": r.get("name"), "run_id": r.get("id"),
    "description": r.get("description"),
    "metadata": _flatten(r.get("metadata")),
    "createdAt": r.get("createdAt"), "updatedAt": r.get("updatedAt"),
} for r in runs])
print(f"{len(df_runs)} run(s)")
df_runs

## 8. Mode B — Runs + outputs + judge scores/reasoning/prompt

For each run item: fetch the main trace, read its `scores[]` (value + reasoning), and
optionally follow each score to its judge **execution trace** to recover the exact prompt
the judge saw plus its raw response. Field selection from cell 2 governs the output shape.

In [ ]:
SELECTED_RUNS = []   # [] = all runs, or e.g. ["baseline-v1", "gpt4o-run"]

target = df_runs["run_name"].tolist() if not SELECTED_RUNS else SELECTED_RUNS
print(f"Pulling {len(target)} run(s)...")

_trace_cache, _obs_cache, _item_cache = {}, {}, {}

def get_run_with_items(run_name):
    return lf_get(f"/api/public/datasets/{quote(DATASET_NAME, safe='')}/runs/{quote(run_name, safe='')}")

def get_trace(trace_id):
    if not trace_id: return None
    if trace_id not in _trace_cache:
        try: _trace_cache[trace_id] = lf_get(f"/api/public/traces/{trace_id}")
        except Exception as e:
            _trace_cache[trace_id] = None; print(f"  ! trace {trace_id}: {e}")
    return _trace_cache[trace_id]

def get_dataset_item(item_id):
    if not item_id: return None
    if item_id not in _item_cache:
        try: _item_cache[item_id] = lf_get(f"/api/public/dataset-items/{item_id}")
        except Exception: _item_cache[item_id] = None
    return _item_cache[item_id]

def get_app_observations(trace_id):
    if not trace_id: return []
    if trace_id not in _obs_cache:
        try: _obs_cache[trace_id] = paginate("/api/public/observations", traceId=trace_id)
        except Exception: _obs_cache[trace_id] = []
    return _obs_cache[trace_id]

def _score_execution_trace_id(score):
    """LLM-judge scores carry a link to their execution trace; field name varies by version."""
    for k in ("executionTraceId", "evalExecutionTraceId", "jobExecutionId", "executionId"):
        if score.get(k): return score.get(k)
    md_ = score.get("metadata") or {}
    if isinstance(md_, dict):
        for k in ("executionTraceId", "traceId", "execution_trace_id"):
            if md_.get(k): return md_.get(k)
    return None

def build_judge_detail(score):
    """Return {value, reasoning, judge_prompt, judge_response} for one score."""
    detail = {
        "name": score.get("name"),
        "value": score.get("value"),
        "string_value": score.get("stringValue"),
        "data_type": score.get("dataType"),
        "reasoning": score.get("comment") if JUDGE_FETCH_REASONING else None,
        "judge_prompt": None,
        "judge_response": None,
        "execution_trace_id": None,
    }
    if JUDGE_FETCH_PROMPT:
        exec_id = _score_execution_trace_id(score)
        detail["execution_trace_id"] = exec_id
        if exec_id:
            et = get_trace(exec_id)
            if et:
                detail["judge_prompt"]   = et.get("input")
                detail["judge_response"] = et.get("output")
    return detail

flat_rows, nested = [], []

for run_name in target:
    run_full = get_run_with_items(run_name)
    run_items = run_full.get("datasetRunItems") or run_full.get("data") or []
    print(f"  run '{run_name}': {len(run_items)} item(s)")
    run_node = {"run": {k: v for k, v in run_full.items()
                        if k not in ("datasetRunItems", "data")}, "items": []}

    for it in run_items:
        trace_id = it.get("traceId")
        item_id  = it.get("datasetItemId")
        di = get_dataset_item(item_id)
        expected = di.get("expectedOutput") if di else None
        trace = get_trace(trace_id)

        # app-side reasoning chain (optional)
        reasoning = None
        if INCLUDE_APP_OBSERVATIONS:
            parts = []
            for o in get_app_observations(trace_id):
                if o.get("output") is not None:
                    parts.append(f"[{o.get('type','')}:{o.get('name','')}] {_flatten(o.get('output'),2000)}")
            reasoning = "\n".join(parts) if parts else None

        # judge scores
        scores = (trace.get("scores") if trace else None) or []
        judge_details = [build_judge_detail(s) for s in scores] if JUDGE_INCLUDE else []

        base = {
            "run_name": run_name,
            "run_item_id": it.get("id"),
            "dataset_item_id": item_id,
            "trace_id": trace_id,
            "observation_id": it.get("observationId"),
            "input": _flatten(trace.get("input") if trace else None, TEXT_LIMIT),
            "actual_output": _flatten(trace.get("output") if trace else None, TEXT_LIMIT),
            "expected_output": _flatten(expected, TEXT_LIMIT),
            "reasoning_chain": _flatten(reasoning, TEXT_LIMIT),
            "n_observations": len(get_app_observations(trace_id)) if INCLUDE_APP_OBSERVATIONS else None,
            "trace_name": trace.get("name") if trace else None,
            "latency_ms": trace.get("latency") if trace else None,
            "total_cost": trace.get("totalCost") if trace else None,
        }
        row = _apply_fields(base)

        # attach judge data per field-selection mode
        if JUDGE_INCLUDE:
            if PER_EVALUATOR_COLUMNS:
                for jd in judge_details:
                    nm = jd["name"] or "score"
                    val = jd["value"] if jd["value"] is not None else jd["string_value"]
                    row[f"score__{nm}"] = val
                    if JUDGE_FETCH_REASONING:
                        row[f"reasoning__{nm}"] = _flatten(jd["reasoning"], TEXT_LIMIT)
                    if JUDGE_FETCH_PROMPT:
                        row[f"judge_prompt__{nm}"]   = _flatten(jd["judge_prompt"], TEXT_LIMIT)
                        row[f"judge_response__{nm}"] = _flatten(jd["judge_response"], TEXT_LIMIT)
            else:
                row["scores_json"] = _flatten(judge_details, TEXT_LIMIT)

        flat_rows.append(row)
        run_node["items"].append({
            "run_item": it, "dataset_item": di, "trace": trace,
            "scores": scores, "judge_details": judge_details,
        })
    nested.append(run_node)

df_run_items = pd.DataFrame(flat_rows)
print(f"\nTotal run items: {len(df_run_items)}")
print("Columns:", list(df_run_items.columns))
df_run_items.head(20)

## 9. Inspect one full record (incl. judge prompt + reasoning)

In [ ]:
if not df_run_items.empty:
    row = df_run_items.iloc[0].to_dict()
    for k, v in row.items():
        s = str(v)
        print(f"--- {k} ---")
        print(s[:2000] + (" ...[more]" if len(s) > 2000 else ""))
        print()
else:
    print("No run items - run cell 8 first.")

## 10. Export to Excel + JSON

Excel/flat-JSON honor your field selection. Nested JSON keeps the full raw archive
(every score, judge execution trace, observation) regardless of toggles.

In [ ]:
EXPORT_DATASETS    = True
EXPORT_ITEMS       = True
EXPORT_RUNS        = True
EXPORT_RUN_ITEMS   = True
EXPORT_NESTED_JSON = True

stamp = _ts(); safe = quote(DATASET_NAME, safe="")

xlsx_path = OUT_DIR / f"{safe}_{stamp}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xl:
    wrote = False
    if EXPORT_DATASETS and 'df_datasets' in dir() and not df_datasets.empty:
        df_datasets.to_excel(xl, sheet_name="datasets", index=False); wrote = True
    if EXPORT_ITEMS and 'df_items' in dir() and not df_items.empty:
        df_items.to_excel(xl, sheet_name="items", index=False); wrote = True
    if EXPORT_RUNS and 'df_runs' in dir() and not df_runs.empty:
        df_runs.to_excel(xl, sheet_name="runs", index=False); wrote = True
    if EXPORT_RUN_ITEMS and 'df_run_items' in dir() and not df_run_items.empty:
        df_run_items.to_excel(xl, sheet_name="run_items", index=False); wrote = True
    if not wrote:
        pd.DataFrame({"note": ["nothing selected/empty"]}).to_excel(xl, sheet_name="empty", index=False)
print("Excel  ->", xlsx_path)

payload = {"exported_at": stamp, "host": LANGFUSE_HOST, "dataset_name": DATASET_NAME,
           "field_selection": FIELDS,
           "judge_config": {"include": JUDGE_INCLUDE, "reasoning": JUDGE_FETCH_REASONING,
                            "prompt": JUDGE_FETCH_PROMPT, "per_evaluator_columns": PER_EVALUATOR_COLUMNS}}
if EXPORT_ITEMS and 'df_items' in dir():
    payload["items"] = df_items.to_dict(orient="records")
if EXPORT_RUNS and 'df_runs' in dir():
    payload["runs"] = df_runs.to_dict(orient="records")
if EXPORT_RUN_ITEMS and 'df_run_items' in dir():
    payload["run_items_flat"] = df_run_items.to_dict(orient="records")

json_path = OUT_DIR / f"{safe}_{stamp}.json"
json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str))
print("JSON   ->", json_path)

if EXPORT_NESTED_JSON and 'nested' in dir() and nested:
    nested_path = OUT_DIR / f"{safe}_{stamp}_nested.json"
    nested_path.write_text(json.dumps(nested, ensure_ascii=False, indent=2, default=str))
    print("Nested ->", nested_path)
print("\nDone.")

---
### Notes

- **Judge reasoning** = `score.comment`. **Judge input prompt** = the `input` of the score's
  *execution trace* (env `langfuse-llm-as-a-judge`), recovered via `JUDGE_FETCH_PROMPT`.
- Execution-trace linkage requires platform **>= 3.119.0** (judge tracing). On older builds,
  `judge_prompt__*` will be empty but score value + reasoning still work.
- The execution-trace id field name has varied across versions; `_score_execution_trace_id`
  checks several. If your `judge_prompt__*` columns come back empty but you know judge traces
  exist, print one raw score (`nested[0]["items"][0]["scores"][0]`) and tell me its keys —
  I'll pin the exact field.
- Turn any column off in **cell 2**. `PER_EVALUATOR_COLUMNS=False` collapses all judge data
  into one `scores_json` cell if you prefer fewer, wider columns.
- `datasetId` item filter is broken upstream — we use `datasetName`.